# Thermal Equilibrium Test

## What is tested

`EvolutionMatrix` and `EvolutionSuper` apply thermal corrections to relaxation rates.
This notebook checks that the corrected rates obey detailed balance, conserve the total
population and have the correct temperature dependence, for both the population rate
matrix and the full Liouville superoperator.

## List of tests

1. `test_driven_rates_accepted`: `EvolutionMatrix` works when only driven rates are given.
2. `test_detailed_balance_two_level`: for a two-level system the rate matrix conserves population, satisfies the Boltzmann ratio and keeps the Boltzmann distribution stationary.
3. `test_temperature_dependence`: downward transitions dominate at low temperature, and the rates become symmetric at high temperature.
4. `test_superoperator_population_block`: the population block of `EvolutionSuper` satisfies the Boltzmann ratio.

## How to run

Run all cells. The last cell calls `run_tests()`. Each test prints one line when it
passes and raises `AssertionError` when it fails.

In [1]:
import os
import sys

import torch

# Makes MaRs importable when the notebook is run from the repository without installation
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

import mars
from mars.population.tr_utils import EvolutionMatrix, EvolutionSuper

dtype = torch.float64

## Tests

In [2]:
def test_driven_rates_accepted():
    """Check that EvolutionMatrix returns a 2x2 matrix when only driven rates are given."""
    energies_K = torch.tensor([-2.0, 2.0], dtype=dtype)
    energies = mars.constants.unit_converter(energies_K, "K_to_Hz")
    temperature = torch.tensor(1.0, dtype=dtype)
    driven_rates = torch.tensor([[0.0, 10.0], [20.0, 0.0]], dtype=dtype)

    evolution = EvolutionMatrix(energies)
    rate_matrix = evolution(temperature, None, driven_rates)

    assert rate_matrix.shape[-2:] == (2, 2), f"Wrong shape of the rate matrix: {rate_matrix.shape}"
    print("Driven rates test passed (EvolutionMatrix accepts driven rates without thermal rates)")


def test_detailed_balance_two_level():
    """Check population conservation, the Boltzmann ratio and the stationary state of a two-level system."""
    energies_K = torch.tensor([-2.0, 2.0], dtype=dtype)
    energies = mars.constants.unit_converter(energies_K, "K_to_Hz")
    temperature = torch.tensor(1.0, dtype=dtype)

    # Symmetric mean rates: k'_01 = k'_10 = 10 1/s
    thermal_rates = torch.tensor([[0.0, 10.0], [10.0, 0.0]], dtype=dtype)

    evolution = EvolutionMatrix(energies, thermal_balance_mode="symmetric")
    rate_matrix = evolution(temperature, thermal_rates=thermal_rates)

    column_sums = rate_matrix.sum(dim=-2)
    assert torch.allclose(column_sums, torch.zeros_like(column_sums), atol=1e-10), (
        f"Column sums are not zero: {column_sums}"
    )

    # E_0 - E_1 = -4 K, so the ratio of the rates is exp(4)
    expected_ratio = torch.exp(-(energies_K[0] - energies_K[1]) / temperature)
    ratio = rate_matrix[0, 1] / rate_matrix[1, 0]
    assert torch.allclose(ratio, expected_ratio, rtol=1e-5), (
        f"Detailed balance violated: got {ratio}, expected {expected_ratio}"
    )

    boltzmann = torch.exp(-energies_K / temperature)
    boltzmann = boltzmann / boltzmann.sum()
    population_change = rate_matrix @ boltzmann
    assert torch.allclose(population_change, torch.zeros_like(population_change), atol=1e-10), (
        f"Boltzmann distribution is not stationary: {population_change}"
    )
    print("Two-level detailed balance test passed (conservation, Boltzmann ratio, stationary state)")


def test_temperature_dependence():
    """Check that downward rates dominate at low temperature and rates are symmetric at high temperature."""
    energies = mars.constants.unit_converter(torch.tensor([0.0, 10.0], dtype=dtype), "K_to_Hz")
    thermal_rates = torch.tensor([[0.0, 5.0], [5.0, 0.0]], dtype=dtype)
    evolution = EvolutionMatrix(energies, thermal_balance_mode="symmetric")

    low_t = evolution(torch.tensor(1.0, dtype=dtype), thermal_rates=thermal_rates)
    assert low_t[0, 1] > 10 * low_t[1, 0], "The downward rate must dominate at low temperature"

    high_t = evolution(torch.tensor(10000.0, dtype=dtype), thermal_rates=thermal_rates)
    ratio = high_t[0, 1] / high_t[1, 0]
    assert torch.abs(ratio - 1.0) < 0.01, f"Rates are not symmetric at high temperature: ratio = {ratio}"
    print("Temperature dependence test passed (downward dominance at low T, symmetry at high T)")


def test_superoperator_population_block():
    """Check that the population block of EvolutionSuper satisfies the Boltzmann ratio."""
    energies_K = torch.tensor([-1.0, 1.0], dtype=dtype)
    energies = mars.constants.unit_converter(energies_K, "K_to_Hz")

    thermal_superop = torch.zeros(4, 4, dtype=dtype)
    thermal_superop[0, 3] = 5.0
    thermal_superop[3, 0] = 0.1

    evolution = EvolutionSuper(energies, thermal_balance_mode="symmetric")
    superop = evolution(
        temp=torch.tensor(1.0, dtype=dtype),
        H=torch.diag(energies).unsqueeze(0),
        thermal_superop=thermal_superop.unsqueeze(0),
    ).squeeze(0)

    # For a two-level system the populations rho_00 and rho_11 have indices 0 and 3
    population_block = superop[[0, 3]][:, [0, 3]]

    expected_ratio = torch.exp(-(energies_K[0] - energies_K[1]) / 1.0)
    ratio = population_block[0, 1] / population_block[1, 0]
    assert torch.allclose(ratio, expected_ratio.to(torch.complex128), rtol=1e-5), (
        f"Detailed balance violated: got {ratio}, expected {expected_ratio}"
    )
    print("Superoperator population block test passed (rates satisfy the Boltzmann ratio)")

## Run all tests

In [3]:
def run_tests():
    """Run every test of this notebook."""
    tests = [
        test_driven_rates_accepted,
        test_detailed_balance_two_level,
        test_temperature_dependence,
        test_superoperator_population_block,
    ]
    for test in tests:
        test()
    print(f"\nAll {len(tests)} thermal equilibrium tests passed.")


run_tests()

Driven rates test passed (EvolutionMatrix accepts driven rates without thermal rates)
Two-level detailed balance test passed (conservation, Boltzmann ratio, stationary state)
Temperature dependence test passed (downward dominance at low T, symmetry at high T)
Superoperator population block test passed (rates satisfy the Boltzmann ratio)

All 4 thermal equilibrium tests passed.
